In [ ]:
import asyncio
import math
import pandas as pd

from itertools import chain, repeat

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [ ]:
from input_output.Standard_Input_and_Output import standard_input, standard_output
from input_output.Class_InputOutput import InputOutput

io = InputOutput()

wb_name = 'IBIT TEMPLATE.xlsx'
cell = 'A1'

wb_name = io.set_xw_book(wb_name)

In [ ]:
from ib_insync import *
from ibkr.Class_IBKR_IB import IBKR_IB
ibkr = IBKR_IB(port=7496)

async def start_ibkr():
    await ibkr.connect()
    print("IBKR connected:", ibkr.ib.isConnected())

await start_ibkr()

In [ ]:
async def make_chains_from_symbols(symbols_list, product_type):

    contracts     = []
    details       = []
    option_chains = []

    for symbol in symbols_list:

        if product_type == 'equity':
            contract = Stock(symbol=symbol, exchange='SMART', currency="USD")
            fut_exch = ""
        elif product_type == "future":
            contract = Future(localSymbol=symbol, exchange='CME', currency="USD")
            fut_exch = "CME"

        contract = await ibkr.ib.qualifyContractsAsync(contract) # this may lead to a printed line since its return has nowhere to be mapped
        contracts.append(contract[0])

        print(contract[0])

        detail = await ibkr.ib.reqContractDetailsAsync(contract[0])
        details.append(detail[0])

        print(detail[0])

        option_chain = await ibkr.ib.reqSecDefOptParamsAsync(underlyingSymbol=detail[0].contract.symbol,
                                                             futFopExchange=fut_exch,
                                                             underlyingSecType=detail[0].contract.secType,
                                                             underlyingConId=detail[0].contract.conId
                                                            )
        option_chains.append(option_chain)

        print(option_chain[0])

        # print('count=', len(stocl_option_chain[0].expirations), stock_option_chain[0].expirations)
        # print('count=', len(stock_option_chain[0].strikes), stock_option_chain[0].strikes)
        # print("2 *", len(stock_option_chain[0].expirations), "*", len(stock_option_chain[0].strikes), "=", 
        #                       len(stock_option_chain[0].expirations) * len(stock_option_chain[0].strikes) * 2)

        # print('\n')

    return option_chains, details, contracts

In [ ]:
stock_symbols = ['IBIT']

s_chain, s_detail, s_contracts = await make_chains_from_symbols(stock_symbols, 'equity')


future_symbols = [
                  'BTCM6',
                  'BTCN6',
                  'BTCQ6',
                  'BTCU6',
                  'BTCV6',
                  'BTCX6',
                  'BTCZ6'
                  ]


f_chain, f_detail, f_contracts = await make_chains_from_symbols(future_symbols, 'future')

combined_zip = chain(zip(s_chain, s_detail, repeat("equity")), zip(f_chain, f_detail, repeat("future")))

In [ ]:
option_contracts = []

for chain, details, product_type in combined_zip:
    
    for expiry in chain[0].expirations:
        for strike in chain[0].strikes:
            for right in ['C', 'P']:
                
                if product_type == 'equity':
                    option_contract = Option(lastTradeDateOrContractMonth=expiry,
                                                    strike=float(strike),
                                                    right=right,
                                                    symbol=details.contract.symbol,
                                                    exchange=details.contract.exchange,
                                                    currency=details.contract.currency,
                                                    )
                
                elif product_type == 'future':
                    option_contract = FuturesOption(lastTradeDateOrContractMonth=expiry,
                                                    strike=float(strike),
                                                    right=right,
                                                    symbol=details.contract.symbol,
                                                    exchange=details.contract.exchange,
                                                    currency=details.contract.currency,
                                                    )
                option_contracts.append(option_contract)

option_contracts = await ibkr.ib.qualifyContractsAsync(*option_contracts) # this may lead to printed lines since its return has nowhere to be mapped

# print(len(option_contracts))

In [ ]:
def clean_num(x):
    """
    Convert IBKR nan / -1 / None style missing values to None.
    """
    if x is None:
        return None
    try:
        if math.isnan(x):
            return None
    except TypeError:
        pass
    if x == -1:
        return None
    return x

In [ ]:
def ticker_row(ticker):
    contract = ticker.contract
    return {
            "conId": getattr(ticker.contract, "conId", None),
            "secType": getattr(contract, "secType", None),
            "symbol": getattr(contract, "symbol", None),
            "expiration": getattr(contract, "lastTradeDateOrContractMonth", None),
            "right": getattr(contract, "right", None),
            "strike": getattr(contract, "strike", None),

            "close": clean_num(getattr(ticker, "close", None)),
        #    "volume": clean_num(getattr(ticker, "volume", None)),
            "avgVolume": clean_num(getattr(ticker, "avVolume", None)),
            "futuresOpenInterest": clean_num(getattr(ticker, "futuresOpenInterest", None)),
            "putOpenInterest": clean_num(getattr(ticker, "putOpenInterest", None)),
            "callOpenInterest": clean_num(getattr(ticker, "callOpenInterest", None)),
        }

In [ ]:
async def get_data(contracts, batch_size=50, wait_seconds=30):
    
    generic_ticks = "100,101,165,588"
    all_rows = []

    for i in range(0, len(contracts), batch_size):
        batch = contracts[i:i + batch_size]
        tickers = []

        print(f"Requesting {i} to {i + len(batch) - 1} of {len(contracts)}")

        for contract in batch:
            ticker = ibkr.ib.reqMktData(
                contract,
                genericTickList=generic_ticks,
                snapshot=False,
            )
            tickers.append(ticker)

        await asyncio.sleep(wait_seconds)

        rows = [ticker_row(ticker) for ticker in tickers]
        all_rows.extend(rows)

        # IMPORTANT: cancel streaming data before next batch
        for ticker in tickers:
            ibkr.ib.cancelMktData(ticker.contract)

        await asyncio.sleep(2)  # small pause between batches

    return pd.DataFrame(all_rows)

In [ ]:
linear_contracts = [*f_contracts, *s_contracts] 
df_linear  = await get_data(linear_contracts, batch_size = 25, wait_seconds=60)
df_linear

In [ ]:
df_options = await get_data(option_contracts, batch_size = 25, wait_seconds=60)
df_options

In [ ]:
df_option_conIds = (
    df_options.pivot_table(
        index=["symbol", "strike"],
        columns=["expiration", "right"],
        values="conId",
        aggfunc="first"
    )
    .sort_index()
    .sort_index(axis=1)
    .reset_index()
)

df_option_conIds

In [ ]:
df_option_prices = (
    df_options.pivot_table(
        index=["symbol", "strike"],
        columns=["expiration", "right"],
        values="close",
        aggfunc="first"
    )
    .sort_index()
    #.sort_index(axis=1)
    .reset_index()
)

df_option_prices

In [ ]:
ws_name = 'linear info'
ws_name, range = io.set_xw_sheet_and_range(wb_name, ws_name, cell)
io.print_xw_df(range, df_linear, headerRows=1)

ws_name = 'option info'
ws_name, range = io.set_xw_sheet_and_range(wb_name, ws_name, cell)
io.print_xw_df(ws_name, range, df_options, headerRows=1)

ws_name = 'option conIds' 
ws_name, range = io.set_xw_sheet_and_range(wb_name, ws_name, cell)
io.print_xw_df(range, df_option_conIds, headerRows=1)

ws_name = 'option prices'
ws_name, range = io.set_xw_sheet_and_range(wb_name, ws_name, cell)
io.print_xw_df(range, df_option_prices, headerRows=1)